# GenAI Newsletter Agent — Class-Based, Extensible Architecture

This notebook implements a **modular, scalable** GenAI newsletter agent with clear separation of concerns:

- `Settings`, `Log`
- `Researcher` (Tavily + RSS)
- `Ranker`
- `LLMClient` (Hugging Face Qwen / OpenAI / Ollama)
- `NewsletterWriter`
- `Agent` Orchestrator
- `Evaluator`
- `MCPServerFactory`


## 1) Dependencies

Uncomment to install as needed.

In [11]:
!pip install --quiet tavily-python feedparser openai mcp python-dotenv tiktoken
!pip install --quiet transformers accelerate sentencepiece bitsandbytes
print('[install] If needed, install the packages by uncommenting the lines above.')

[install] If needed, install the packages by uncommenting the lines above.


## 2) Settings & Logger

In [39]:
import os
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Tuple
from datetime import datetime, timedelta
import re, pathlib
from pathlib import Path
os.environ['TAVILY_API_KEY'] = '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'

# -------------------------
# 2) Settings & Logger
# -------------------------

@dataclass
class Settings:
    """Central configuration for the GenAI Newsletter Agent.

    Environment variables:
        LLM_PROVIDER: Provider selector. One of {"huggingface", "openai", "ollama"}.
            Default: "huggingface".
        HUGGINGFACE_MODEL: HF model id to load locally (e.g., "Qwen/Qwen2.5-1.5B-Instruct").
        OPENAI_API_KEY: Key for OpenAI-compatible APIs.
        OPENAI_BASE_URL: Base URL for OpenAI-compatible backends (e.g., vLLM server).
            Default: "https://api.openai.com/v1".
        OPENAI_MODEL: Model id for OpenAI-compatible backends.
        OLLAMA_HOST: Ollama HTTP host (default: "http://localhost:11434").
        OLLAMA_MODEL: Ollama model tag (e.g., "qwen2.5:1.5b-instruct").
        TAVILY_API_KEY: API key for Tavily web research (optional).
        LLM_TEMPERATURE: Default generation temperature (float).
        LLM_MAX_TOKENS: Default max new tokens for generation (int).
        DEBUG: "1" to enable debug logs.

    Notes:
        - You can override any of these in-notebook using `os.environ[...]` and
          re-instantiating `Settings()`.
    """

    # LLM provider selection
    llm_provider: str = os.getenv("LLM_PROVIDER", "huggingface")

    # Hugging Face (transformers)
    huggingface_model: str = os.getenv("HUGGINGFACE_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")

    # OpenAI-compatible
    openai_api_key: str = os.getenv("OPENAI_API_KEY", "")
    openai_base_url: str = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
    openai_model: str = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

    # Ollama (local)
    ollama_host: str = os.getenv("OLLAMA_HOST", "http://localhost:11434")
    ollama_model: str = os.getenv("OLLAMA_MODEL", "qwen2.5:1.5b-instruct")

    # Research
    tavily_api_key: str = os.getenv("TAVILY_API_KEY", "")

    # Generation knobs
    default_temperature: float = float(os.getenv("LLM_TEMPERATURE", "0.4"))
    default_max_tokens: int = int(os.getenv("LLM_MAX_TOKENS", "1500"))

    # Logging
    debug: bool = os.getenv("DEBUG", "0") == "1"


class Log:
    """Tiny logger for consistent, toggleable console output.

    This is intentionally minimal so you can swap it with `logging` later.

    Attributes:
        enabled: Global on/off for all logs.
        debug_enabled: Controls whether debug logs are printed.
    """

    enabled: bool = True
    debug_enabled: bool = False

    @classmethod
    def configure(cls, enabled: bool = True, debug: bool = False) -> None:
        """Configure logging behavior.

        Args:
            enabled: Master switch for all logs.
            debug: If True, enable debug-level logs too.
        """
        cls.enabled = enabled
        cls.debug_enabled = debug

    @classmethod
    def debug(cls, *args: Any) -> None:
        """Print a debug-level log line if debug is enabled."""
        if cls.enabled and cls.debug_enabled:
            print("[DEBUG]", *args)

    @classmethod
    def info(cls, *args: Any) -> None:
        """Print an info-level log line if logging is enabled."""
        if cls.enabled:
            print("[INFO]", *args)

    @classmethod
    def warn(cls, *args: Any) -> None:
        """Print a warning-level log line if logging is enabled."""
        if cls.enabled:
            print("[WARN]", *args)


SETTINGS = Settings()
Log.configure(enabled=True, debug=SETTINGS.debug)
Log.info("LLM_PROVIDER:", SETTINGS.llm_provider)
if SETTINGS.llm_provider == "huggingface":
    Log.info("HUGGINGFACE_MODEL:", SETTINGS.huggingface_model)
elif SETTINGS.llm_provider == "openai":
    Log.info("OPENAI_BASE_URL:", SETTINGS.openai_base_url, "OPENAI_MODEL:", SETTINGS.openai_model)
else:
    Log.info("OLLAMA_HOST:", SETTINGS.ollama_host, "OLLAMA_MODEL:", SETTINGS.ollama_model)

[INFO] LLM_PROVIDER: huggingface
[INFO] HUGGINGFACE_MODEL: Qwen/Qwen2.5-1.5B-Instruct


## 3) Utilities

In [40]:
# -------------------------
# 3) Utilities
# -------------------------

class Utils:
    """General utility functions for time windows, cleaning, and file I/O."""

    @staticmethod
    def time_window(days: int = 30) -> Tuple[str, str]:
        """Return ISO strings for (start, end) covering the last `days` days.

        Args:
            days: Number of days in the past to include.

        Returns:
            (start_iso, end_iso) representing the time window in UTC.
        """
        end = datetime.utcnow()
        start = end - timedelta(days=days)
        return start.isoformat(), end.isoformat()

    @staticmethod
    def clean_text(s: Optional[str]) -> str:
        """Collapse whitespace and trim a string safely.

        Args:
            s: Arbitrary string (can be None).

        Returns:
            A normalized string with consecutive whitespace reduced to a single space.
        """
        return re.sub(r"\s+", " ", (s or "")).strip()

    @staticmethod
    def save_markdown(text: str, filename: str = "newsletter.md") -> str:
        """Save text to a Markdown file.

        Args:
            text: Markdown content.
            filename: Output file name.

        Returns:
            Absolute path to the saved file.
        """
        path = Path(filename).resolve()
        path.write_text(text, encoding="utf-8")
        Log.info("[save] Wrote markdown to:", str(path))
        return str(path)

## 4) Researcher (Tavily + RSS)

In [41]:
# -------------------------
# 4) Researcher
# -------------------------

class Researcher:
    """Collect recent GenAI items from multiple sources (Tavily + RSS).

    Methods return items normalized to:
        {
          "title": str,
          "url": str,
          "content": str,        # compact summary or snippet
          "published_at": str,   # if available
        }
    """

    def __init__(self, settings: Settings) -> None:
        """Initialize a Researcher.

        Args:
            settings: Global Settings object; used for API keys and params.
        """
        self.s = settings

    def from_tavily(self, query: str, days: int = 30, max_results: int = 10) -> List[Dict[str, Any]]:
        """Query Tavily for relevant links.

        Args:
            query: Search query string (you can include time filters as text).
            days: Lookback window in days.
            max_results: Maximum number of results to fetch.

        Returns:
            A list of normalized items. Returns [] if missing API key or if errors occur.
        """
        if not self.s.tavily_api_key:
            Log.debug("[research:tavily] No API key; skipping.")
            return []

        try:
            from tavily import TavilyClient
            tc = TavilyClient(api_key=self.s.tavily_api_key)
            start, _ = Utils.time_window(days)
            q = f"{query} after:{start[:10]}"
            Log.info("[research:tavily] query =", q)
            res = tc.search(q, max_results=max_results)
            items: List[Dict[str, Any]] = []
            for r in res.get("results", []):
                items.append({
                    "title": r.get("title"),
                    "url": r.get("url"),
                    "content": r.get("content") or r.get("snippet") or "",
                    "published_at": r.get("published_time") or "",
                })
            Log.info(f"[research:tavily] fetched={len(items)}")
            return items
        except Exception as e:
            Log.warn("[research:tavily] Error:", e)
            return []

    def from_rss(self, days: int = 30, limit: int = 30) -> List[Dict[str, Any]]:
        """Parse curated RSS feeds for recent items.

        Args:
            days: Lookback window (days).
            limit: Max entries to consider per feed.

        Returns:
            List of normalized items; [] if `feedparser` is unavailable.
        """
        try:
            import feedparser, time as _t
        except Exception:
            Log.warn("[research:rss] feedparser not installed; skipping.")
            return []

        feeds = [
            "https://openai.com/blog/rss.xml",
            "https://research.google/blog/feed/",
            "https://stability.ai/blog.rss",
            "https://ai.googleblog.com/feeds/posts/default",
            "https://huggingface.co/blog/feed.xml",
        ]
        start_iso, _ = Utils.time_window(days)
        start_ts = datetime.fromisoformat(start_iso).timestamp()

        out: List[Dict[str, Any]] = []
        for url in feeds:
            Log.info("[research:rss] Fetch:", url)
            try:
                fp = feedparser.parse(url)
                for e in fp.entries[:limit]:
                    pub_ts = None
                    if getattr(e, "published_parsed", None):
                        pub_ts = _t.mktime(e.published_parsed)
                    elif getattr(e, "updated_parsed", None):
                        pub_ts = _t.mktime(e.updated_parsed)
                    if pub_ts and pub_ts >= start_ts:
                        out.append({
                            "title": e.get("title", ""),
                            "url": e.get("link", ""),
                            "content": Utils.clean_text(e.get("summary", "")),
                            "published_at": e.get("published", e.get("updated", "")),
                        })
            except Exception as e:
                Log.warn("[research:rss] Error:", e)
        Log.info("[research:rss] collected=", len(out))
        return out

    def gather(self, days: int = 30, interests: Optional[List[str]] = None, max_per_interest: int = 6) -> List[Dict[str, Any]]:
        """Gather and deduplicate items from Tavily + RSS for given interest topics.

        Args:
            days: Lookback period in days.
            interests: Topic seeds to bias web research. If None, a default set is used.
            max_per_interest: Per-topic max results for Tavily.

        Returns:
            Deduplicated list of items across sources.
        """
        if not interests:
            interests = ["LLM", "multimodal", "agents", "open source", "safety"]

        items: List[Dict[str, Any]] = []
        for topic in interests:
            items.extend(self.from_tavily(f"Generative AI {topic} advancements", days=days, max_results=max_per_interest))
        if len(items) < 6:
            Log.info("[research] Tavily sparse → RSS fallback")
            items.extend(self.from_rss(days=days))

        # Dedupe by URL
        seen = set()
        dedup: List[Dict[str, Any]] = []
        for it in items:
            u = it.get("url", "")
            if u and u not in seen:
                seen.add(u)
                dedup.append(it)
        Log.info("[research] total_unique=", len(dedup))
        return dedup

## 5) Ranker

In [42]:
# -------------------------
# 5) Ranker
# -------------------------

class Ranker:
    """Score and rank research items with a simple, transparent heuristic.

    Scoring favors longer content and presence of domain keywords. Replace this
    with embeddings, learned rankers, or recency-weighted schemes later.
    """

    def __init__(self, keywords: Optional[List[str]] = None) -> None:
        """Create a Ranker.

        Args:
            keywords: Domain-relevant terms to upweight. If None, a default set is used.
        """
        self.keywords = keywords or [
            "benchmark", "release", "paper", "framework", "dataset",
            "open source", "capability", "reasoning", "agent", "multimodal",
        ]

    def score(self, item: Dict[str, Any]) -> int:
        """Compute a simple relevance score for one item.

        Args:
            item: Item dict with at least a "content" field.

        Returns:
            Integer score; higher is better.
        """
        c = (item.get("content") or "").lower()
        s = len(c)
        s += sum(5 for kw in self.keywords if kw in c)
        return s

    def top_k(self, items: List[Dict[str, Any]], k: int = 12) -> List[Dict[str, Any]]:
        """Return the top-k ranked items.

        Args:
            items: Candidate items to rank.
            k: Number of items to return.

        Returns:
            Items sorted by score (desc) with length ≤ k.
        """
        ranked = sorted(items, key=self.score, reverse=True)
        Log.info(f"[rank] ranked={len(ranked)}; returning top_k={k}")
        return ranked[:k]

## 6) LLM Client (Hugging Face / OpenAI / Ollama)

In [43]:
# -------------------------
# 6) LLM Client
# -------------------------

class LLMClient:
    """Provider-agnostic chat completion client.

    Supported backends:
        - Hugging Face Transformers (e.g., **Qwen**)
        - OpenAI / OpenAI-compatible chat.completions
        - Ollama local HTTP server

    Methods:
        complete(): Perform a chat-style completion and return text.
    """

    def __init__(self, settings: Settings) -> None:
        """Initialize an LLMClient.

        Args:
            settings: Global Settings object.
        """
        self.s = settings

    def complete(
        self,
        system_prompt: str,
        user_prompt: str,
        model: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
    ) -> str:
        """Dispatch chat completion to the selected provider.

        Args:
            system_prompt: System message content for the LLM.
            user_prompt: User message content for the LLM.
            model: Optional provider-specific model override.
            temperature: Sampling temperature; if omitted, uses Settings.default_temperature.
            max_tokens: Max new tokens; if omitted, uses Settings.default_max_tokens.

        Returns:
            Assistant's text reply. On provider errors, returns a helpful error string.
        """
        provider = (self.s.llm_provider or "huggingface").lower()
        temperature = self.s.default_temperature if temperature is None else float(temperature)
        max_tokens = self.s.default_max_tokens if max_tokens is None else int(max_tokens)
        model = model or (self.s.huggingface_model if provider == "huggingface" else None)

        if provider == "huggingface":
            return self._complete_huggingface(system_prompt, user_prompt, model, temperature, max_tokens)
        elif provider == "ollama":
            return self._complete_ollama(system_prompt, user_prompt, model, temperature, max_tokens)
        else:
            return self._complete_openai(system_prompt, user_prompt, model, temperature, max_tokens)

    def _complete_huggingface(self, system_prompt: str, user_prompt: str, model: str,
                              temperature: float, max_tokens: int) -> str:
        """Run generation using Hugging Face Transformers (Qwen and friends).

        Notes:
            - Uses a simple text-generation pipeline. For production-quality chat,
              apply model-specific chat templates.
            - Falls back to a mocked response if `transformers` is unavailable.

        Returns:
            Generated text or an error/mock message.
        """
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
            import torch
        except Exception:
            Log.warn("[llm:hf] transformers not installed — returning mocked output.")
            return "[MOCKED HF]\n" + user_prompt[:800]

        try:
            Log.info(f"[llm:hf] Loading model: {model}")
            tok = AutoTokenizer.from_pretrained(model)
            mdl = AutoModelForCausalLM.from_pretrained(
                model,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto",
            )
            pipe = pipeline("text-generation", model=mdl, tokenizer=tok, device_map="auto")
            prompt = f"System: {system_prompt}\nUser: {user_prompt}\nAssistant:"
            out = pipe(
                prompt,
                max_new_tokens=int(max_tokens),
                temperature=float(temperature),
                do_sample=True,
                top_p=0.9,
            )[0]["generated_text"]
            reply = out.split("Assistant:", 1)[-1].strip()
            return reply or "[EMPTY RESPONSE FROM HF MODEL]"
        except Exception as e:
            Log.warn("[llm:hf] Error:", e)
            return f"[ERROR: HuggingFace] {e}"

    def _complete_openai(self, system_prompt: str, user_prompt: str, model: Optional[str],
                         temperature: float, max_tokens: int) -> str:
        """Call an OpenAI-compatible chat.completions endpoint.

        Returns:
            Generated text or a mocked response if no API key is configured.
        """
        if not self.s.openai_api_key:
            Log.warn("[llm:openai] Missing OPENAI_API_KEY — returning mocked output.")
            return "[MOCKED OPENAI]\n" + user_prompt[:800]

        try:
            from openai import OpenAI
            client = OpenAI(api_key=self.s.openai_api_key, base_url=self.s.openai_base_url)
            mdl = model or self.s.openai_model
            Log.info(f"[llm:openai] model={mdl} base_url={self.s.openai_base_url}")
            resp = client.chat.completions.create(
                model=mdl,
                temperature=float(temperature),
                max_tokens=int(max_tokens),
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            return resp.choices[0].message.content
        except Exception as e:
            Log.warn("[llm:openai] Error:", e)
            return f"[ERROR: OpenAI] {e}"

    def _complete_ollama(self, system_prompt: str, user_prompt: str, model: Optional[str],
                         temperature: float, max_tokens: int) -> str:
        """Call a local Ollama chat endpoint.

        Returns:
            Generated text or an error string if the HTTP call fails.
        """
        import json, requests
        mdl = model or self.s.ollama_model
        url = f"{self.s.ollama_host.rstrip('/')}/api/chat"
        payload = {
            "model": mdl,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "stream": False,
            "options": {"temperature": float(temperature), "num_predict": int(max_tokens)},
        }
        try:
            Log.info(f"[llm:ollama] POST {url} model={mdl}")
            resp = requests.post(url, json=payload, timeout=120)
            resp.raise_for_status()
            data = resp.json()
            return (data.get("message", {}) or {}).get("content", "")
        except Exception as e:
            Log.warn("[llm:ollama] Error:", e)
            return f"[ERROR: Ollama] {e}"

## 7) NewsletterWriter

In [44]:
# -------------------------
# 7) NewsletterWriter
# -------------------------

class NewsletterWriter:
    """Compose a structured Markdown newsletter from ranked items using an LLM."""

    SYSTEM_PROMPT = (
        "You are a precise, engaging AI editor. Write a concise Markdown newsletter about recent "
        "Generative AI advancements. Use neutral, evidence-based language. Include inline source links. "
        "Prefer bullets and short paragraphs."
    )

    def __init__(self, llm: LLMClient) -> None:
        """Create a NewsletterWriter.

        Args:
            llm: An LLMClient instance used to generate the newsletter.
        """
        self.llm = llm

    def write(self, items: List[Dict[str, Any]], persona: str, tone: str = "analytical", region: str = "global") -> str:
        """Generate newsletter Markdown for the given items and persona.

        Args:
            items: Ranked research items (title/url/content).
            persona: Target reader persona (e.g., "Staff ML engineer at an enterprise").
            tone: Writing tone (e.g., "analytical", "practical").
            region: Regional focus ("global", "APAC", etc.).

        Returns:
            Markdown string containing the newsletter.
        """
        src_lines: List[str] = []
        for it in items:
            title = it.get("title", "Untitled")
            url = it.get("url", "")
            note = Utils.clean_text(it.get("content", ""))[:300]
            src_lines.append(f"- {title} | {url} | {note}")

        user_prompt = (
            f"Persona: {persona}\nTone: {tone}\nRegion: {region}\n\n"
            "Sources (title | url | note):\n" + "\n".join(src_lines) + "\n\n"
            "Write a Markdown newsletter with: Title+date, 3–6 themed sections, each item as a 1–2 sentence summary "
            "with a link; a short 'What it means' wrap-up for the persona; and a small 'Further Reading' list."
        )
        Log.info("[writer] Calling LLM for newsletter synthesis…")
        return self.llm.complete(self.SYSTEM_PROMPT, user_prompt, max_tokens=2000)

## 8) Agent Orchestrator

In [45]:
# -------------------------
# 8) Agent Orchestrator
# -------------------------

class Agent:
    """High-level orchestrator for the GenAI newsletter pipeline.

    Pipeline:
        research → rank → write → save

    Attributes:
        s: Global settings.
        researcher: Researcher component.
        ranker: Ranker component.
        llm: LLMClient component.
        writer: NewsletterWriter component.
    """

    def __init__(self, settings: Settings) -> None:
        """Construct the agent and its components.

        Args:
            settings: Global Settings.
        """
        self.s = settings
        self.researcher = Researcher(settings)
        self.ranker = Ranker()
        self.llm = LLMClient(settings)
        self.writer = NewsletterWriter(self.llm)

    def run(
        self,
        days: int = 30,
        interests: Optional[List[str]] = None,
        persona: str = "Staff ML engineer",
        tone: str = "analytical",
        region: str = "global",
        filename: str = "newsletter.md",
    ) -> Dict[str, Any]:
        """Execute the full pipeline and save the newsletter to disk.

        Args:
            days: Research lookback window in days.
            interests: Topic seeds for research; if None, a default set is used.
            persona: Target persona for the newsletter.
            tone: Writing tone.
            region: Regional focus.
            filename: Output Markdown filename.

        Returns:
            Dict with:
                - "path": str, path to the saved Markdown file
                - "picked": List[Dict[str, Any]], the ranked items used
        """
        Log.info(f"[agent] Starting pipeline: days={days} tone={tone} region={region}")
        items = self.researcher.gather(days=days, interests=interests)
        if not items:
            Log.warn("[agent] No items found; using mock examples.")
            items = [
                {"title": "Example: New Multimodal Model Release", "url": "https://example.com/mm",
                 "content": "A powerful multimodal model with better grounding and tool-use."},
                {"title": "Example: Efficient Fine-Tuning", "url": "https://example.com/ft",
                 "content": "Method reduces compute by ~40% on adaptation benchmarks."},
            ]

        top = self.ranker.top_k(items, k=12)
        md = self.writer.write(top, persona=persona, tone=tone, region=region)
        path = Utils.save_markdown(md, filename=filename)
        return {"path": path, "picked": top}

## 9) Evaluator

In [46]:
# -------------------------
# 9) Evaluator
# -------------------------

class Evaluator:
    """Lightweight sanity checks for generated Markdown content."""

    @staticmethod
    def sanity_check_markdown(path: str) -> Dict[str, Any]:
        """Detect basic newsletter issues (headers/links/bullets).

        Args:
            path: Path to a generated Markdown file.

        Returns:
            {"ok": bool, "issues": List[str]} summarizing whether it passed checks.
        """
        text = pathlib.Path(path).read_text(encoding="utf-8")
        issues: List[str] = []
        if "# " not in text:
            issues.append("Missing H1 title")
        if "http" not in text:
            issues.append("No links detected")
        if "- " not in text and "* " not in text:
            issues.append("No bullet lists detected")
        Log.info("[eval] Issues:", issues or "None")
        return {"ok": len(issues) == 0, "issues": issues}

Log.info("[core] Docstring-rich classes loaded.")

[INFO] [core] Docstring-rich classes loaded.


## 10) Demo Run

In [47]:
agent = Agent(SETTINGS)
out = agent.run(days=30, interests=['LLM','agents','multimodal'], filename='newsletter_demo_v2.md')
print('[demo] saved at', out['path'])
print('[demo] picked sample', [{k:v for k,v in p.items() if k in ('title','url')} for p in out['picked'][:3]])
print(Evaluator.sanity_check_markdown(out['path']))


[INFO] [agent] Starting pipeline: days=30 tone=analytical region=global
[INFO] [research:tavily] query = Generative AI LLM advancements after:2025-09-15


/tmp/ipython-input-2897312064.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


[INFO] [research:tavily] fetched=6
[INFO] [research:tavily] query = Generative AI agents advancements after:2025-09-15
[INFO] [research:tavily] fetched=6
[INFO] [research:tavily] query = Generative AI multimodal advancements after:2025-09-15
[INFO] [research:tavily] fetched=6
[INFO] [research] total_unique= 14
[INFO] [rank] ranked=14; returning top_k=12
[INFO] [writer] Calling LLM for newsletter synthesis…
[INFO] [llm:hf] Loading model: Qwen/Qwen2.5-1.5B-Instruct


Device set to use cuda:0


[INFO] [save] Wrote markdown to: /content/newsletter_demo_v2.md
[demo] saved at /content/newsletter_demo_v2.md
[demo] picked sample [{'title': 'Contemporary AI Trends -AI, Generative AI, LLMs, Agentic ...', 'url': 'https://medium.com/@adnanmasood/contemporary-ai-trends-ai-generative-ai-llms-agentic-ai-and-evals-definitions-evidence-699a42172d07'}, {'title': 'Insights on Artificial Intelligence - QuantumBlack', 'url': 'https://www.mckinsey.com/capabilities/quantumblack/our-insights'}, {'title': 'The Latest AI News and AI Breakthroughs that Matter Most', 'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates'}]
[INFO] [eval] Issues: None
{'ok': True, 'issues': []}
